### For single file

In [12]:
import xml.etree.ElementTree as ET
import json
import os

def xml_to_qupath_geojson(xml_path, geojson_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    features = []

    for i, annotation in enumerate(root.findall(".//Annotation")):
        # 1. Get label
        label = annotation.attrib.get("Name", "Tumor")
        
        coords = []
        for coord in annotation.findall(".//Coordinate"):
            x = float(coord.attrib["X"])
            y = float(coord.attrib["Y"])
            coords.append([x, y])

        if not coords:
            continue

        # 2. Ensure polygon is closed
        if coords[0] != coords[-1]:
            coords.append(coords[0])

        # 3. Use the MOST compatible QuPath structure
        feature = {
            "type": "Feature",
            "id": f"annotation_{i}",
            "geometry": {
                "type": "Polygon",
                "coordinates": [coords]
            },
            "properties": {
                "objectType": "annotation",
                "classification": {
                    "name": label,
                    "color": [255, 0, 0] # Explicit RGB list
                }
            }
        }
        features.append(feature)

    geojson = {
        "type": "FeatureCollection",
        "features": features
    }

    # Use ensure_ascii=False to prevent encoding errors
    with open(geojson_path, "w", encoding='utf-8') as f:
        json.dump(geojson, f, indent=2, ensure_ascii=False)

    print(f"File saved to: {os.path.abspath(geojson_path)}")
# example usage
xml_file = "/data_64T_3/Dataset/CAMELYON16/annotations_xml/test_011.xml"
geojson_file = "test_011.geojson"

xml_to_qupath_geojson(xml_file, geojson_file)

File saved to: /home/rajaj/Project/test_011.geojson


### For multiple files

In [14]:
import xml.etree.ElementTree as ET
import json
import os
import glob

def xml_to_qupath_geojson(xml_path, geojson_path):
    """Converts a single XML file to a QuPath-compatible GeoJSON."""
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        features = []

        # Find all Annotation tags in the XML [cite: 1, 8]
        for i, annotation in enumerate(root.findall(".//Annotation")):
            # Get the label name from XML attributes [cite: 1, 9]
            label = annotation.attrib.get("Name", f"Annotation {i}")
            
            coords = []
            # Extract individual X, Y coordinates [cite: 1, 2, 3]
            for coord in annotation.findall(".//Coordinate"):
                x = float(coord.attrib["X"])
                y = float(coord.attrib["Y"])
                coords.append([x, y])

            if not coords:
                continue

            # Ensure the polygon is closed (last point matches first) [cite: 1, 2, 7, 8]
            if coords[0] != coords[-1]:
                coords.append(coords[0])

            # Build the GeoJSON feature structure for QuPath [cite: 1, 8]
            feature = {
                "type": "Feature",
                "id": f"annotation_{i}",
                "geometry": {
                    "type": "Polygon",
                    "coordinates": [coords] # Nested list required for Polygons [cite: 1, 8]
                },
                "properties": {
                    "objectType": "annotation",
                    "classification": {
                        "name": label, # Fixes PathClass parse error [cite: 8]
                        "color": [255, 0, 0] 
                    }
                }
            }
            features.append(feature)

        geojson = {
            "type": "FeatureCollection",
            "features": features
        }

        with open(geojson_path, "w", encoding='utf-8') as f:
            json.dump(geojson, f, indent=2, ensure_ascii=False)
            
        return len(features)
    except Exception as e:
        print(f"Error processing {xml_path}: {e}")
        return 0

def run_batch_processing(input_dir, output_dir):
    """Creates the output folder and processes all XML files found."""
    # 1. Create output folder if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created output directory: {output_dir}")

    # 2. Get list of all XML files
    xml_files = glob.glob(os.path.join(input_dir, "*.xml"))
    print(f"Found {len(xml_files)} files to process.")

    # 3. Loop through files
    for xml_path in xml_files:
        # Get the filename without the extension (e.g., 'test_011')
        base_name = os.path.splitext(os.path.basename(xml_path))[0]
        output_path = os.path.join(output_dir, f"{base_name}.geojson")
        
        num_annotations = xml_to_qupath_geojson(xml_path, output_path)
        print(f"Processed {base_name}: {num_annotations} annotations saved.")

# --- CONFIGURE PATHS HERE ---
input_dir = "/data_64T_3/Dataset/CAMELYON16/annotations_xml"
output_dir = "/data_64T_3/Dataset/CAMELYON16/annotations_geojson"

run_batch_processing(input_dir, output_dir)

Created output directory: /data_64T_3/Dataset/CAMELYON16/annotations_geojson
Found 160 files to process.
Processed tumor_089: 116 annotations saved.
Processed test_040: 78 annotations saved.
Processed tumor_080: 12 annotations saved.
Processed tumor_105: 160 annotations saved.
Processed tumor_055: 29 annotations saved.
Processed tumor_034: 91 annotations saved.
Processed tumor_096: 42 annotations saved.
Processed tumor_014: 13 annotations saved.
Processed tumor_031: 86 annotations saved.
Processed tumor_060: 1 annotations saved.
Processed test_075: 53 annotations saved.
Processed tumor_092: 3 annotations saved.
Processed test_097: 1 annotations saved.
Processed tumor_106: 228 annotations saved.
Processed tumor_076: 38 annotations saved.
Processed tumor_005: 4 annotations saved.
Processed tumor_067: 1 annotations saved.
Processed tumor_066: 3 annotations saved.
Processed tumor_016: 9 annotations saved.
Processed tumor_020: 22 annotations saved.
Processed tumor_037: 26 annotations saved.